In [3]:
from dotenv import load_dotenv
import os

load_dotenv()  # 현재 작업 경로를 기준으로 .env를 로드
api_key = os.getenv("GOOGLE_API_KEY")

In [4]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


model = init_chat_model("gemini-2.5-flash", timeout = 30,api_key = api_key,model_provider="google_genai",temperature=0.7)

agent = create_agent(model, tools=[])

In [ ]:
# ModelRequest 속성
# ModelRequest는 에이전트의 모델 호출 정보를 담는 데이터 클래스로, 미들웨어에서 요청을 검사하고 수정할 때 사용됩니다. override() 메서드를 통해 여러 속성을 동시에 변경할 수 있습니다.

# 아래 코드는 ModelRequest를 사용하여 동적으로 모델과 시스템 프롬프트를 변경하는 예시입니다.

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# 기본 모델과 고급 모델 정의
basic_model = init_chat_model("gemini-2.5-flash", timeout = 30,api_key = api_key,model_provider="google_genai",temperature=0.7)
advanced_model = init_chat_model("gemini-2.5-pro", timeout = 30,api_key = api_key,model_provider="google_genai",temperature=0.7)


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """대화 복잡도에 따라 모델 선택"""
    message_count = len(request.state["messages"])

    # 긴 대화에는 고급 모델 사용
    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    request.model = model
    print(f"모델 선택: {request.model.model}")
    return handler(request)


agent = create_agent(
    model=basic_model, tools=[], middleware=[dynamic_model_selection]  # 기본 모델
)

In [8]:
from langchain_core.messages import HumanMessage

def stream_graph(graph, inputs):
    for event in graph.stream(inputs, stream_mode="values"):
        message = event["messages"][-1]
        if hasattr(message, "pretty_print"):
            message.pretty_print()
        else:
            print(message)

stream_graph(
    agent,
    inputs={
        "messages": [HumanMessage(content="머신러닝의 동작 원리에 대해서 설명해줘")]
    },
)

================================ Human Message =================================

머신러닝의 동작 원리에 대해서 설명해줘
모델 선택: gemini-2.5-flash


/var/folders/px/2gqqh24s6yb9zp3q9m6th5f80000gn/T/ipykernel_30383/2023401703.py:19: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


================================== Ai Message ==================================

머신러닝(Machine Learning)은 컴퓨터가 **데이터로부터 학습**하여 특정 작업을 수행하거나 예측을 할 수 있도록 하는 인공지능의 한 분야입니다. 핵심 원리는 "명시적으로 프로그래밍하지 않고도 컴퓨터가 학습하게 하는 것"입니다.

간단한 비유로 설명해볼게요. 어린아이가 자전거 타는 법을 배우는 과정과 비슷합니다.
*   **데이터:** 수없이 많은 시행착오 (넘어지고, 다시 일어서고, 균형을 잡으려는 시도).
*   **모델:** 아이의 뇌와 몸 (학습하는 주체).
*   **학습:** 시행착오를 통해 균형을 잡는 방법, 페달을 밟는 강도 등을 스스로 조절하는 과정.
*   **예측/수행:** 결국 넘어지지 않고 자전거를 탈 수 있게 되는 것.

이제 머신러닝의 동작 원리를 좀 더 단계별로 자세히 설명해 드릴게요.

---

### 머신러닝의 동작 원리 5단계

머신러닝 모델이 작동하는 과정은 크게 5가지 단계로 나눌 수 있습니다.

**1. 데이터 준비 (Data Preparation)**
*   **무엇인가?** 머신러닝의 가장 기본이자 핵심입니다. 모델이 학습할 수 있도록 양질의 데이터를 수집하고 정제하는 단계입니다.
*   **어떻게 하는가?**
    *   **데이터 수집:** 문제 해결에 필요한 다양한 데이터를 모읍니다. (예: 주택 가격 예측을 위해 주택의 크기, 방 개수, 위치, 건축 연도 등)
    *   **데이터 전처리:** 수집된 데이터는 종종 불완전하거나 노이즈가 많습니다. 누락된 값을 채우고, 오류를 수정하고, 형식을 통일하며, 모델이 이해하기 쉬운 형태로 변환합니다.
    *   **데이터 분할:** 학습용(Training Set), 검증용(Validation Set), 테스트용(Test Set)으로 데이터를 나눕니다. 일반적으로 7:1:2 또는 8:2 비율로 나눕니다.
 

In [9]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """대화 복잡도에 따라 모델 선택"""
    message_count = len(request.state["messages"][-1].content)
    print(f"글자수: {message_count}")

    # 긴 대화에는 고급 모델 사용
    if message_count > 10:
        # 여러 속성 동시 변경
        new_request = request.override(
            model=advanced_model,
            system_prompt="emoji 를 사용해서 답변해줘",
            tool_choice="auto",
        )
    else:
        new_request = request.override(
            model=basic_model,
            system_prompt="한 문장으로 간결하게 답변해줘. emoji 는 사용하지 말아줘.",
            tool_choice="auto",
        )
    print(f"모델 선택: {new_request.model.model}")
    return handler(new_request)

    
agent = create_agent(
    model=basic_model, tools=[], middleware=[dynamic_model_selection]  # 기본 모델
)

In [10]:
stream_graph(agent, inputs={"messages": [HumanMessage(content="머신러닝 동작원리")]})

================================ Human Message =================================

머신러닝 동작원리
글자수: 9
모델 선택: gemini-2.5-flash
================================== Ai Message ==================================

데이터에서 패턴과 규칙을 학습하여 새로운 데이터에 대한 예측이나 결정을 수행합니다.


In [ ]:
# Pydantic 모델로 입력 스키마 정의
# 복잡한 입력 검증이 필요한 경우 Pydantic 모델을 사용하여 명확한 입력 스키마를 정의할 수 있습니다. args_schema 매개변수에 Pydantic 모델을 전달하면, 도구 호출 시 자동으로 입력 검증이 수행됩니다.

# Pydantic 모델을 사용하면 다음과 같은 이점이 있습니다:

# 타입 검증 및 변환 자동화
# Literal 타입을 통한 허용값 제한
# Field의 description을 통한 상세한 매개변수 설명
# 아래 코드는 Pydantic 모델을 사용한 날씨 조회 도구 예시입니다.

In [ ]:

# Literal 타입을 통한 허용값 제한
# Field의 description을 통한 상세한 매개변수 설명

from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool

class WeatherInput(BaseModel):
    """Input for weather queries."""

    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius", description="Temperature unit preference"
    )
    include_forecast: bool = Field(default=False, description="Include 5-day forecast")


@tool(args_schema=WeatherInput)
def get_weather(
    location: str, units: str = "celsius", include_forecast: bool = False
) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"현재 {location} 지역의 날씨는 {temp} {units[0].upper()} 도"
    if include_forecast:
        result += "\n다음 5일 날씨: 맑음"
    return result

In [14]:
# 도구 테스트
print(
    get_weather.invoke(
        {"location": "Seoul", "units": "celsius", "include_forecast": True}
    )
)

현재 Seoul 지역의 날씨는 22 C 도
다음 5일 날씨: 맑음


In [ ]:
# 컨텍스트 접근
# 도구는 에이전트 상태, 런타임 컨텍스트 및 장기 메모리에 액세스할 수 있을 때 가장 강력합니다. 이를 통해 도구는 컨텍스트 인식 결정을 내리고, 응답을 개인화하며, 대화 전반에 걸쳐 정보를 유지할 수 있습니다.

# ToolRuntime 매개변수를 통해 다음 런타임 정보에 액세스할 수 있습니다:

# 속성	설명
# state	실행을 통해 흐르는 변경 가능한 데이터 (메시지, 카운터, 커스텀 필드)
# context	사용자 ID, 세션 세부 정보 등 불변 구성 정보
# store	대화 전반에 걸친 영구 장기 메모리
# stream_writer	도구 실행 중 커스텀 업데이트 스트리밍
# config	실행을 위한 RunnableConfig
# tool_call_id	현재 도구 호출의 고유 ID
# ToolRuntime 사용
# ToolRuntime을 사용하면 단일 매개변수로 모든 런타임 정보에 액세스할 수 있습니다. 도구 시그니처에 runtime: ToolRuntime을 추가하면 LLM에는 노출되지 않고 자동으로 주입됩니다.

# runtime.state를 통해 현재 그래프 상태에 접근하고, runtime.context를 통해 컨텍스트 정보에 접근할 수 있습니다.

# 아래 코드는 ToolRuntime을 사용하여 상태와 컨텍스트에 접근하는 도구 예시입니다.